# QLoRA training notebook

Ноутбук использует общий training pipeline из `qlora/src`, а не дублирует загрузку модели внутри себя. Базовая модель сохраняется локально в `data/models/huggingface/...`.

In [1]:
# Базовые импорты, определение корня проекта и загрузка qlora-конфига.
from __future__ import annotations

import random
import sys
from pathlib import Path

import torch

project_root = Path.cwd()
while project_root.name != 'smart-therm-bot' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from qlora.src.dataset import load_examples
from qlora.src.formatting import encode_example
from qlora.src.modeling import create_peft_config, load_tokenizer, prepare_local_base_model
from qlora.src.paths import artifact_paths, dataset_path
from qlora.src.pipeline import finalize_training_cycle
from qlora.src.training import run_training
from qlora.src.validation import validate_artifacts
from src.config.qlora_models import QLoRAWorkspaceConfig

config = QLoRAWorkspaceConfig.load()
random.seed(config.seed)
torch.manual_seed(config.seed)
config


QLoRAWorkspaceConfig(project_root=PosixPath('/Users/andrey/study/smart-therm-bot'), data_dir='data', models_dir='data/models', mode='full_train', enabled=False, base_model='Qwen/Qwen2.5-3B', dataset_path='data/processed/chat/lora_pairs.jsonl', tmp_dir='qlora/tmp', max_seq_length=2048, seed=42, backend=QLoRABackendConfig(device_preference='auto', trust_remote_code=True, use_mps_bitsandbytes=True), qlora=QLoRAQuantizationConfig(r=8, alpha=16, dropout=0.05, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype='float16'), training=QLoRATrainingConfig(epochs=3, batch_size=1, gradient_accumulation_steps=4, learning_rate=0.0001, warmup_ratio=0.03, logging_steps=1, save_total_limit=2), export=QLoRAExportConfig(output_root='qlora/artifacts', adapter_dir_name='adapter', merged_dir_name='merged', gguf_dir_name='gguf', ollama_dir_name='ollama', merge_after_train=True, gguf_enabled=True, gg

In [2]:
# Подготовка путей и гарантированное сохранение базовой модели локально.
config = QLoRAWorkspaceConfig.load()
DATASET_PATH = dataset_path(config)
ARTIFACTS = artifact_paths(config)
ALL_EXAMPLES = load_examples(DATASET_PATH)
BASE_MODEL = prepare_local_base_model(config)

print('dataset:', DATASET_PATH)
print('examples:', len(ALL_EXAMPLES))
print('base_model_id:', BASE_MODEL.model_id)
print('base_model_dir:', BASE_MODEL.snapshot_dir)
print('metadata_path:', BASE_MODEL.metadata_path)
print('downloaded_now:', BASE_MODEL.downloaded_now)
validate_artifacts(config)


dataset: /Users/andrey/study/smart-therm-bot/data/processed/chat/lora_pairs.jsonl
examples: 716
base_model_id: Qwen/Qwen2.5-3B
base_model_dir: /Users/andrey/study/smart-therm-bot/data/models/huggingface/Qwen%2FQwen2.5-3B
metadata_path: /Users/andrey/study/smart-therm-bot/data/models/huggingface/Qwen%2FQwen2.5-3B/.snapshot-info.json
downloaded_now: False


{'adapter_exists': True,
 'merged_exists': True,
 'gguf_exists': False,
 'modelfile_exists': False}

In [3]:
# Выбор режима и предпросмотр того, что пойдет в обучение.
config = QLoRAWorkspaceConfig.load()
NOTEBOOK_MODE = config.mode
print('NOTEBOOK_MODE =', NOTEBOOK_MODE)

if NOTEBOOK_MODE == 'micro_test':
    SELECTED_EXAMPLES = ALL_EXAMPLES[: config.micro_test.sample_pairs]
    EFFECTIVE_EPOCHS = config.micro_test.epochs
elif NOTEBOOK_MODE == 'full_train':
    SELECTED_EXAMPLES = ALL_EXAMPLES
    EFFECTIVE_EPOCHS = config.training.epochs
else:
    SELECTED_EXAMPLES = ALL_EXAMPLES
    EFFECTIVE_EPOCHS = config.training.epochs

print('selected_examples:', len(SELECTED_EXAMPLES))
print('effective_epochs:', EFFECTIVE_EPOCHS)
SELECTED_EXAMPLES[:2]


NOTEBOOK_MODE = full_train
selected_examples: 716
effective_epochs: 3


[QLoRAExample(instruction='Что означает пометка KNX у версии SmartTherm на Ozon и отличаются ли версии по функционалу?', response='Пометка KNX в карточке Ozon была артефактом интерфейса маркетплейса, а не признаком отдельного KNX-интерфейса у контроллера. По функционалу версии не отличались, разница была только в наличии датчиков в комплекте.'),
 QLoRAExample(instruction='Нужно ли прошивать SmartTherm перед использованием и можно ли управлять им удалённо из интернета?', response='Контроллер уже прошивается и проверяется продавцом на его котле. В штатной прошивке облачного управления нет. Для удалённого доступа можно либо перепрошить устройство на вариант с облаком, либо заходить на домашний компьютер удалённо и управлять уже через него. Для Home Assistant упоминались два варианта OpenTherm-подключения: через ESPHome и через Tasmota.')]

In [4]:
# Проверка tokenizer и пример encoded sample через общий helper-код.
config = QLoRAWorkspaceConfig.load()
tokenizer = load_tokenizer(config)
peft_config = create_peft_config(config)
ENCODED_EXAMPLES = [
    encode_example(example, tokenizer, max_seq_length=config.max_seq_length)
    for example in SELECTED_EXAMPLES
]

print('pad_token:', tokenizer.pad_token)
print('target_modules:', peft_config.target_modules)
print('encoded_examples:', len(ENCODED_EXAMPLES))
ENCODED_EXAMPLES[:1]


pad_token: <|endoftext|>
target_modules: {'o_proj', 'q_proj', 'gate_proj', 'k_proj', 'up_proj', 'v_proj', 'down_proj'}
encoded_examples: 716


[{'input_ids': [16664,
   510,
   72819,
   24634,
   8215,
   129974,
   26909,
   75301,
   8178,
   13132,
   31292,
   55,
   13932,
   63262,
   2247,
   46173,
   15770,
   1001,
   4195,
   13073,
   35604,
   263,
   7587,
   127888,
   132039,
   58095,
   63262,
   2247,
   46173,
   17686,
   54713,
   73728,
   129070,
   1939,
   2582,
   510,
   16854,
   12228,
   8178,
   13132,
   31292,
   55,
   5805,
   129297,
   24634,
   6020,
   52571,
   35604,
   263,
   129760,
   20396,
   2184,
   50527,
   19355,
   15869,
   93766,
   96951,
   19355,
   21032,
   128647,
   126064,
   4793,
   8178,
   125741,
   21032,
   128647,
   11,
   20396,
   18658,
   36305,
   130812,
   20264,
   131255,
   38800,
   31292,
   55,
   12,
   142796,
   19355,
   21032,
   128647,
   13932,
   132161,
   3038,
   90783,
   13,
   126206,
   54713,
   73728,
   129070,
   63262,
   2247,
   46173,
   18658,
   127888,
   134096,
   11,
   38379,
   131692,
   129760,
   73626,
  

In [5]:
# Основной этап QLoRA-обучения. Использует только общий код из qlora/src и локальную копию модели.
config = QLoRAWorkspaceConfig.load()
train_result = None

if NOTEBOOK_MODE == 'micro_test':
    train_result = run_training(
        config,
        examples_limit=config.micro_test.sample_pairs,
        epochs_override=config.micro_test.epochs,
    )
elif NOTEBOOK_MODE == 'full_train':
    train_result = run_training(config)
else:
    print('inspect mode: training skipped')

if train_result is not None:
    print('adapter_path:', train_result.adapter_path)
    print('sample_count:', train_result.sample_count)
    print('base_model_dir:', train_result.base_model.snapshot_dir)
    print('downloaded_now:', train_result.base_model.downloaded_now)

train_result


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The module name Qwen%2FQwen2_dot_5_hyphen_3B (originally Qwen%2FQwen2.5-3B) is not a valid Python identifier. Please rename the original module to avoid import issues.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/andrey/study/smart-therm-bot/.venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,1.996920
2,1.869852
3,2.241269
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


/Users/andrey/study/smart-therm-bot/.venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/Users/andrey/study/smart-therm-bot/.venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the

adapter_path: /Users/andrey/study/smart-therm-bot/qlora/artifacts/adapter
sample_count: 716
base_model_dir: /Users/andrey/study/smart-therm-bot/data/models/huggingface/Qwen%2FQwen2.5-3B
downloaded_now: False


TrainRunResult(adapter_path=PosixPath('/Users/andrey/study/smart-therm-bot/qlora/artifacts/adapter'), sample_count=716, base_model=LocalBaseModelInfo(model_id='Qwen/Qwen2.5-3B', snapshot_dir=PosixPath('/Users/andrey/study/smart-therm-bot/data/models/huggingface/Qwen%2FQwen2.5-3B'), metadata_path=PosixPath('/Users/andrey/study/smart-therm-bot/data/models/huggingface/Qwen%2FQwen2.5-3B/.snapshot-info.json'), ready=True, downloaded_now=False))

In [9]:
# Merge/export/smoke-check через общий pipeline без повторного запуска training.
config = QLoRAWorkspaceConfig.load()
pipeline_result = None

if train_result is not None:
    smoke_examples = SELECTED_EXAMPLES[: config.micro_test.sample_pairs] if NOTEBOOK_MODE == 'micro_test' else SELECTED_EXAMPLES
    pipeline_result = finalize_training_cycle(config, train_result, examples=smoke_examples)
else:
    print('inspect mode: export skipped')

result = {
    'mode': NOTEBOOK_MODE,
    'base_model_dir': str(BASE_MODEL.snapshot_dir),
    'base_model_metadata': str(BASE_MODEL.metadata_path),
    'downloaded_now': BASE_MODEL.downloaded_now,
    'adapter_path': str(pipeline_result.adapter_path) if pipeline_result else None,
    'merged_path': str(pipeline_result.merged_path) if pipeline_result and pipeline_result.merged_path else None,
    'gguf_path': str(pipeline_result.gguf_path) if pipeline_result and pipeline_result.gguf_path else None,
    'modelfile_path': str(pipeline_result.modelfile_path) if pipeline_result and pipeline_result.modelfile_path else None,
    'sample_count': pipeline_result.sample_count if pipeline_result else 0,
    'artifact_status': pipeline_result.artifact_status if pipeline_result else validate_artifacts(config),
}
result


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The module name Qwen%2FQwen2_dot_5_hyphen_3B (originally Qwen%2FQwen2.5-3B) is not a valid Python identifier. Please rename the original module to avoid import issues.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:hf-to-gguf:Loading model: merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> Q8_0, shape = {2048, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> Q8_0, shape = {11008, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> Q8_0, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> Q8_0, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> Q8_0, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float16 

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{'mode': 'full_train',
 'base_model_dir': '/Users/andrey/study/smart-therm-bot/data/models/huggingface/Qwen%2FQwen2.5-3B',
 'base_model_metadata': '/Users/andrey/study/smart-therm-bot/data/models/huggingface/Qwen%2FQwen2.5-3B/.snapshot-info.json',
 'downloaded_now': False,
 'adapter_path': '/Users/andrey/study/smart-therm-bot/qlora/artifacts/adapter',
 'merged_path': '/Users/andrey/study/smart-therm-bot/qlora/artifacts/merged',
 'gguf_path': '/Users/andrey/study/smart-therm-bot/qlora/artifacts/gguf/model-q8_0.gguf',
 'modelfile_path': '/Users/andrey/study/smart-therm-bot/qlora/artifacts/ollama/Modelfile',
 'sample_count': 716,
 'artifact_status': {'adapter_exists': True,
  'merged_exists': True,
  'gguf_exists': True,
  'modelfile_exists': True}}